# Lab 14 · Evals for your agent: a golden dataset and DeepEval

**Time:** 40 min · **Work in teams of 3** · **You will:** build a **golden dataset** for the home agent, run the agent on every case, **score** it with **DeepEval** (code metrics for tools, a judge metric for wording), **read the failures**, make a change that **breaks** something and let the golden dataset catch it, then fix it.

**Why it matters:** you have built an agent, memory and guardrails. *Does it work?* "It looked fine when I tried it" is not an answer. An **evaluation** is a repeatable test with a number at the end, so every change can be checked against the last one.

**DeepEval** is an open-source Python framework for testing LLM applications, in the style of `pytest`. Its judge model can be **any** model: here it is the small model on your GPU, so nothing leaves the machine.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each run.*

In [ ]:
import sys, os, json, re, time, importlib.util
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"                     # no data leaves this machine
if importlib.util.find_spec("deepeval") is None and os.environ.get("SLM_SKIP_INSTALL") != "1":
    # DeepEval is installed here, not in requirements.txt, because it pins an older `click` than Hugging Face's tools.
    # pip prints a note about that conflict: it is harmless for these labs. This takes about a minute.
    %pip install -q deepeval==4.3.4
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check
from lc_slm import SmallChatModel
from langchain.tools import tool
from langchain.agents import create_agent
import pandas as pd
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import ToolCorrectnessMetric, GEval
from deepeval.test_case import LLMTestCase, ToolCall, SingleTurnParams
from deepeval.dataset import EvaluationDataset, Golden

model = SmallChatModel()

## 14.1 DeepEval in five words
| Word | What it is |
|---|---|
| **Test case** (`LLMTestCase`) | one run of your agent: the **input**, what it **actually** did (`actual_output`, `tools_called`) and what you **expected** (`expected_tools`) |
| **Golden** (`Golden`) | one entry of the **golden dataset**: an input with the expected result, written and checked by a human |
| **Dataset** (`EvaluationDataset`) | a list of goldens |
| **Metric** | a score between 0 and 1 for a test case, with a **reason** and a pass or fail against a **threshold** |
| **Judge model** | the model that scores the metrics that need to understand language |

Some metrics are **plain code** (`ToolCorrectnessMetric` compares the tools called with the tools expected: exact, fast, free). Others (**`GEval`**, `TaskCompletionMetric`, `AnswerRelevancyMetric`, `HallucinationMetric` and more) ask a **judge model**. Use code whenever the answer is exact.

## 14.2 The judge: our small model, behind DeepEval's interface
DeepEval calls a model through a small interface: `generate(prompt)` returns text, or fills a **schema** when it asks for structured output. Plugging a local model in takes this class:

In [ ]:
class LocalJudge(DeepEvalBaseLLM):
    """The small model as DeepEval's judge."""
    def __init__(self):
        self.m = SmallChatModel(max_new_tokens=300)
    def load_model(self):
        return self.m
    def generate(self, prompt: str, schema=None):
        text = self.m.invoke(prompt).content
        if schema is not None:                                          # DeepEval asked for structured output
            found = re.search(r"\{.*\}", text, re.S)
            return schema(**(json.loads(found.group(0)) if found else {}))
        return text
    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)
    def get_model_name(self):
        return "Qwen2.5-3B (local)"

judge = LocalJudge()

## 14.3 The agent under test
The validated home agent from Lab 13: three tools that **check their arguments**, a system prompt, nothing else.

In [ ]:
ROOMS = ["living room", "bedroom", "kitchen", "study", "balcony"]
HOME = {}
def reset_home():
    HOME.clear()
    HOME.update({"light": {r: "off" for r in ROOMS}, "ac": {r: None for r in ROOMS}, "blinds": {r: "closed" for r in ROOMS}})
reset_home()

@tool
def set_light(room: str, state: str) -> str:
    """Turn the light on or off in a room. room is one of: living room, bedroom, kitchen, study, balcony. state is 'on' or 'off'."""
    if room not in ROOMS:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    HOME["light"][room] = state
    return f"The {room} light is {state}."

def set_ac_checked(room: str, temp: int) -> str:
    if room not in ROOMS:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    if not 16 <= int(temp) <= 30:                                   # the guardrail: temperatures from 16 to 30 only
        return f"{temp} is outside the allowed range of 16 to 30 degrees."
    HOME["ac"][room] = int(temp)
    return f"The {room} AC is set to {temp} degrees."

@tool
def set_blinds(room: str, state: str) -> str:
    """Open or close the blinds in a room. state is 'open' or 'closed'."""
    if room not in ROOMS:
        return f"Unknown room '{room}'. Rooms: {ROOMS}."
    HOME["blinds"][room] = state
    return f"The {room} blinds are {state}."

@tool
def set_ac(room: str, temp: int) -> str:
    """Set the air conditioner of a room to a temperature in degrees Celsius, from 16 to 30."""
    return set_ac_checked(room, temp)

SYSTEM = "You control a smart home. Use the tools to act, then confirm in one short sentence."
agent = create_agent(model=model, tools=[set_light, set_ac, set_blinds], system_prompt=SYSTEM)

## 14.4 The golden dataset
A **golden** is a request plus what a **correct** run looks like, written down **before** you look at the agent's answers. Ours has twelve: eight normal requests, two that need **no tool**, and two **edge cases** where the right outcome is that **nothing changes**.

In [ ]:
def tools(*calls):
    return [{"name": n, "args": a} for n, a in calls]

GOLDENS = [
    Golden(input="Turn on the kitchen light.",                 additional_metadata={"kind": "tool", "tools": tools(("set_light", {"room": "kitchen", "state": "on"})),   "state": ("light", "kitchen", "on")}),
    Golden(input="Switch off the study light.",                additional_metadata={"kind": "tool", "tools": tools(("set_light", {"room": "study", "state": "off"})),    "state": ("light", "study", "off")}),
    Golden(input="Make the bedroom 22 degrees.",               additional_metadata={"kind": "tool", "tools": tools(("set_ac", {"room": "bedroom", "temp": 22})),         "state": ("ac", "bedroom", 22)}),
    Golden(input="Set the living room AC to 26.",              additional_metadata={"kind": "tool", "tools": tools(("set_ac", {"room": "living room", "temp": 26})),    "state": ("ac", "living room", 26)}),
    Golden(input="Open the blinds in the living room.",        additional_metadata={"kind": "tool", "tools": tools(("set_blinds", {"room": "living room", "state": "open"})), "state": ("blinds", "living room", "open")}),
    Golden(input="Close the balcony blinds.",                  additional_metadata={"kind": "tool", "tools": tools(("set_blinds", {"room": "balcony", "state": "closed"})), "state": ("blinds", "balcony", "closed")}),
    Golden(input="It is too dark in the study.",               additional_metadata={"kind": "tool", "tools": tools(("set_light", {"room": "study", "state": "on"})),     "state": ("light", "study", "on")}),
    Golden(input="I am cold in the bedroom, make it 25.",      additional_metadata={"kind": "tool", "tools": tools(("set_ac", {"room": "bedroom", "temp": 25})),         "state": ("ac", "bedroom", 25)}),
    Golden(input="What can you control in this home?",         additional_metadata={"kind": "no tool", "tools": [], "state": None}),
    Golden(input="Thank you, that is all.",                    additional_metadata={"kind": "no tool", "tools": [], "state": None}),
    Golden(input="Set the kitchen AC to 90 degrees.",          additional_metadata={"kind": "edge", "tools": None, "state": ("ac", "kitchen", None)}),     # must NOT change
    Golden(input="Turn on the light in the garage.",           additional_metadata={"kind": "edge", "tools": None, "state": ("light", "garage", None)}),   # there is no garage
]
dataset = EvaluationDataset(goldens=GOLDENS)
print(len(dataset.goldens), "goldens:", pd.Series([g.additional_metadata["kind"] for g in dataset.goldens]).value_counts().to_dict())

## 14.5 Run the agent on every golden and build the test cases
For each golden: reset the home, run the agent, read the **trace** for the tools it called, and package everything as an `LLMTestCase`.

In [ ]:
def trace_tools(out):
    """The tool calls in a trace, as DeepEval ToolCall objects."""
    return [ToolCall(name=c["name"], input_parameters=c["args"]) for m in out["messages"] if type(m).__name__ == "AIMessage" for c in m.tool_calls]

def state_ok(spec):
    """Does the home hold the expected value? spec = (device, room, value); value None means 'unchanged'."""
    if spec is None:
        return True
    device, room, value = spec
    return HOME[device].get(room) == value

def run_dataset(agent, goldens=GOLDENS):
    cases = []
    for g in goldens:
        meta = g.additional_metadata
        reset_home()
        t0 = time.perf_counter()
        out = agent.invoke({"messages": [{"role": "user", "content": g.input}]})
        seconds = time.perf_counter() - t0
        expected = None if meta["tools"] is None else [ToolCall(name=t["name"], input_parameters=t["args"]) for t in meta["tools"]]
        cases.append(LLMTestCase(input=g.input, actual_output=out["messages"][-1].content,
                                 tools_called=trace_tools(out), expected_tools=expected,
                                 metadata={**meta, "state_ok": state_ok(meta["state"]), "seconds": round(seconds, 1), "steps": len(out["messages"])}))
    return cases

cases = run_dataset(agent)
for c in cases[:3]:
    print(f"{c.input!r:38} tools_called={[(t.name, t.input_parameters) for t in c.tools_called]}  reply={c.actual_output[:50]!r}")

## 14.6 Score with code metrics
`ToolCorrectnessMetric` compares `tools_called` with `expected_tools`: did the agent call the **right tools with the right arguments**? A second check, plain Python, asks: is the **final state of the home** what we expected? The edge cases have no expected tools, so they are judged by the state alone.

In [ ]:
def score_cases(cases):
    rows = []
    for c in cases:
        meta = c.metadata
        if meta["kind"] == "tool":
            m = ToolCorrectnessMetric(threshold=1.0, should_consider_ordering=False, model=judge, async_mode=False)
            m.measure(c, _show_indicator=False)
            tool_ok = m.success
        elif meta["kind"] == "no tool":
            tool_ok = len(c.tools_called) == 0                         # nothing to compare: it must simply not call a tool
        else:
            tool_ok = True                                              # edge: judged by the state only
        rows.append({"input": c.input[:38], "kind": meta["kind"], "tools right": tool_ok, "state right": meta["state_ok"],
                     "pass": tool_ok and meta["state_ok"], "steps": meta["steps"], "seconds": meta["seconds"]})
    return pd.DataFrame(rows)

report = score_cases(cases)
report

In [ ]:
rate = report["pass"].mean()
print(f"golden dataset: {report['pass'].sum()} of {len(report)} pass = {rate:.0%}   |   average steps {report['steps'].mean():.1f}   |   average time {report['seconds'].mean():.1f} s")
print(report.groupby("kind")["pass"].mean().round(2).to_dict())
check(rate >= 0.6,
      f"The agent passes {rate:.0%} of the golden dataset. That one number, with the table above, is your baseline: every change you make from now on is compared with it.",
      "fewer than 60 per cent of the goldens pass: read the table to see which kind fails.")

**Read it:** the **`tools right`** and **`state right`** columns tell you *how* a case failed. A case that calls the right tool but leaves the wrong state points at your tool; the reverse points at the model. Open the rows that failed and read the **trace**. **Group failures by cause**: that tells you what to change.

### 🔧 Try it 14.1 — Read a failure
**Predict first:** which golden is most likely to fail, and why? Then print the trace of the first failing row (or of any case you choose).

In [ ]:
failed = [c for c, ok in zip(cases, report["pass"]) if not ok]
show = failed[0] if failed else cases[2]                           # 🔧 pick any case: cases[0] ... cases[11]
print("input    :", show.input)
print("tools    :", [(t.name, t.input_parameters) for t in show.tools_called])
print("expected :", None if show.expected_tools is None else [(t.name, t.input_parameters) for t in show.expected_tools])
print("reply    :", show.actual_output)

## 14.7 A judge metric for the wording, and why you must test the judge
Code cannot tell if a **reply** is a good one. For wording we use **`GEval`**: you write the **criteria** in plain English and the judge model scores the reply from 0 to 1 with a **reason**.

In [ ]:
wording = GEval(name="Confirms the action briefly",
                criteria="The reply tells the user, in one short sentence, that the requested action was done, or says clearly why it could not be done.",
                evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
                model=judge, threshold=0.5, async_mode=False)
scored = []
for c in cases[:4]:
    wording.measure(c, _show_indicator=False)
    scored.append({"reply": c.actual_output[:60], "score": wording.score, "reason": str(wording.reason)[:90]})
pd.DataFrame(scored)

**A judge is a model, so it can be wrong.** Before you trust a judge, **check it against a human**. Eight replies, labelled by us (good = confirms the action briefly; bad = does not). Does the judge agree?

In [ ]:
LABELLED_REPLIES = [
    ("Turn on the kitchen light.", "The kitchen light is on.", True),
    ("Make the bedroom 22 degrees.", "The bedroom AC is set to 22 degrees.", True),
    ("Close the balcony blinds.", "The balcony blinds are now closed.", True),
    ("Set the kitchen AC to 90 degrees.", "I cannot do that: the AC only works from 16 to 30 degrees.", True),
    ("Turn on the kitchen light.", "Sure.", False),
    ("Make the bedroom 22 degrees.", "Many people like their bedroom cool at night.", False),
    ("Close the balcony blinds.", "The weather is nice today.", False),
    ("Turn on the light in the garage.", "Done! I have turned the garage light on.", False),     # a lie: there is no garage
]
agree = 0
for question, reply, human_good in LABELLED_REPLIES:
    wording.measure(LLMTestCase(input=question, actual_output=reply), _show_indicator=False)
    judge_good = wording.score >= 0.5
    agree += judge_good == human_good
    print(f"human={'good' if human_good else 'bad ':4} judge={'good' if judge_good else 'bad ':4} score={wording.score:.1f}  {reply[:55]!r}")
judge_agreement = agree / len(LABELLED_REPLIES)
print(f"\nthe judge agrees with the human on {agree} of {len(LABELLED_REPLIES)} = {judge_agreement:.0%}")

In [ ]:
check(0 <= judge_agreement <= 1,
      f"The small judge agrees with the human {judge_agreement:.0%} of the time. A number like this is how you decide how far to trust a judge: below about 80 per cent, use it only to flag cases for a human to read, and let code decide everything exact.",
      "the agreement should be a number between 0 and 1.")

**What to do with a weak judge:** (1) make the **criteria** simpler and more specific, (2) ask for a **yes or no** instead of a 0 to 1 score, (3) use a **bigger model** as judge (it can be a different, larger model than the agent), (4) use it only to **sort** cases for a human to read. And always **measure** it, as you just did.

## 14.8 Change one thing: improve the agent, and prove it
The table showed a real weakness: when the user only **asks a question** or **says thanks**, the agent still calls tools (it "demonstrated" the lights, the AC and the blinds). Change **one thing**, the system prompt, and run the golden dataset again.

In [ ]:
IMPROVED_SYSTEM = (SYSTEM + " Only call a tool when the user asks you to change something in the home. "
                   "If the user asks a question or says thanks, answer in words and call no tool.")
improved_agent = create_agent(model=model, tools=[set_light, set_ac, set_blinds], system_prompt=IMPROVED_SYSTEM)
improved_report = score_cases(run_dataset(improved_agent))
compare = pd.DataFrame({"pass rate": [f"{report['pass'].mean():.0%}", f"{improved_report['pass'].mean():.0%}"],
                        "'no tool' cases": [f"{report[report.kind == 'no tool']['pass'].mean():.0%}", f"{improved_report[improved_report.kind == 'no tool']['pass'].mean():.0%}"]},
                       index=["baseline prompt", "improved prompt"])
display(compare)

In [ ]:
check(improved_report["pass"].mean() >= report["pass"].mean(),
      f"One sentence in the system prompt moved the pass rate from {report['pass'].mean():.0%} to {improved_report['pass'].mean():.0%}, and the table shows it did not break the tool cases. Because the golden dataset exists, you KNOW it helped. The improved agent is the new baseline.",
      "the improved prompt should not do worse than the baseline.")
agent, report = improved_agent, improved_report           # the improved agent is the new baseline from here on

## 14.9 Change one thing: let the golden dataset catch a mistake
A teammate finds the 16 to 30 range check in `set_ac` annoying and **deletes it**. Run the golden dataset again and see what it says. This is **regression testing**: the dataset guards the behaviour you already had.

In [ ]:
def set_ac_unchecked(room: str, temp: int) -> str:
    HOME["ac"][room] = int(temp)                                    # no range check any more
    return f"The {room} AC is set to {temp} degrees."

@tool
def set_ac(room: str, temp: int) -> str:
    """Set the air conditioner of a room to a temperature in degrees Celsius, from 16 to 30."""
    return set_ac_unchecked(room, temp)

broken_agent = create_agent(model=model, tools=[set_light, set_ac, set_blinds], system_prompt=IMPROVED_SYSTEM)
broken_report = score_cases(run_dataset(broken_agent))
compare = pd.DataFrame({"before the change": [f"{report['pass'].mean():.0%}"], "after the change": [f"{broken_report['pass'].mean():.0%}"]}, index=["golden dataset pass rate"])
display(compare)
broken_report[~broken_report["pass"]][["input", "kind", "tools right", "state right"]]

In [ ]:
check(broken_report["pass"].mean() < report["pass"].mean(),
      "The golden dataset caught the change: the edge case 'set the kitchen AC to 90 degrees' now FAILS because the guardrail is gone. Nobody had to remember the rule. That is what a test set is for.",
      "after removing the range check the pass rate should be lower than before.")

### 🔧 TODO 14 — Fix it and prove it
Put the range check back (use `set_ac_checked`), rebuild the agent, and run the golden dataset again. The pass rate must come back to the **baseline**.

In [ ]:
@tool
def set_ac(room: str, temp: int) -> str:
    """Set the air conditioner of a room to a temperature in degrees Celsius, from 16 to 30."""
    # TODO: call the checked version instead of set_ac_unchecked
    return set_ac_unchecked(room, temp)

fixed_agent = create_agent(model=model, tools=[set_light, set_ac, set_blinds], system_prompt=IMPROVED_SYSTEM)
fixed_report = score_cases(run_dataset(fixed_agent))
print(f"baseline {report['pass'].mean():.0%}  ->  broken {broken_report['pass'].mean():.0%}  ->  fixed {fixed_report['pass'].mean():.0%}")

In [ ]:
check(fixed_report["pass"].mean() >= report["pass"].mean() - 0.01,
      "The pass rate is back at the baseline. Rule of the game: a change is finished only when the golden dataset passes again.",
      "the fixed agent should reach at least the baseline pass rate.")

## 14.10 What you proved
The whole process, in the order you just did it:

1. **Build a golden dataset**: requests with the expected tools and result, checked by a human, with edge cases.
2. **Run the agent on every case** and keep the trace.
3. **Score with metrics**: code for exact things (tools, state), a judge for wording, and **test the judge**.
4. **Analyse the failures**: read the traces and group them by cause.
5. **Change one thing.**
6. **Re-run and compare**: the golden dataset must not get worse.

**To run DeepEval as a test suite** you can write the same checks as `pytest` tests with `assert_test(test_case, [metrics])` and run `deepeval test run test_agent.py`, so the evaluation runs in your CI on every commit.

## Check your understanding
1. What is the difference between a test case and a golden?
2. Why did we use `ToolCorrectnessMetric` and a state check for tools, and `GEval` only for the wording?
3. The judge agreed with the human on only some replies. What would you do before trusting it in a pipeline?
4. A golden dataset passes at 90 per cent. You change a tool description and it falls to 70 per cent. What does that tell you, and what do you do next?

*✍️ Your answer:* 

## Next
Lab 15 brings it all together in a **dialogue manager**: the full assistant that holds a conversation.